# 04 — ML Attrition Risk

**Layer:** Gold (writes back) · **Day:** 2 · **Authoritative script:** `scripts/day2_train_attrition_model.py`

## Objective

Train an attrition-risk model on the silver employee table, evaluate it, generate SHAP explanations, and write per-employee risk scores back to the gold layer so Power BI and the FastAPI Copilot can consume them.

## Inputs

- `lakehouse/silver/employees.parquet`

## Outputs

- `lakehouse/gold/fact_attrition_risk.parquet`
  - `EmployeeID`, `RiskScore` (0–1), `RiskBand` (`Low`/`Medium`/`High`)
  - `TopDriver1` … `TopDriver3` and corresponding signed `*Impact` SHAP values
- `apps/api/models/attrition_rf.joblib` — Random Forest pipeline
- `apps/api/models/attrition_logit.joblib` — Logistic Regression baseline pipeline
- `apps/api/models/feature_meta.joblib` — feature names, num/cat columns, decision threshold
- `apps/api/models/shap_explainer.joblib` — SHAP TreeExplainer
- `apps/api/models/day2_model_metrics.json` — evaluation metrics
- `docs/images/shap_summary.png` — global SHAP summary plot

## Approach

1. **Split** — stratified 75 / 25 train/test on `AttritionFlag`.
2. **Pipeline** — `ColumnTransformer(StandardScaler + OneHotEncoder)` → classifier.
3. **Models** — Random Forest (primary) and Logistic Regression (interpretable baseline).
4. **Threshold** — chosen by maximising F1 on the precision-recall curve, not the default 0.5.
5. **Risk band** — `Low < 0.25 ≤ Medium < 0.50 ≤ High`.
6. **Explainability** — `shap.TreeExplainer` on the fitted RF; the top three signed contributions per employee are written into the risk table.

## Business value

The model is **decision support**: it gives HR Business Partners and managers a rank-ordered list of employees and a plain-English explanation of why each was ranked. Combined with the policy Q&A and board-narrative endpoints, this is what makes the Copilot useful in a CHRO setting rather than just another classifier.

## Reproduce

```powershell
python scripts\day2_train_attrition_model.py
python scripts\verify_day2.py
```

In [ ]:
from pathlib import Path
import json
import pandas as pd

METRICS = Path('../apps/api/models/day2_model_metrics.json')
RISK = Path('../lakehouse/gold/fact_attrition_risk.parquet')
print('metrics exist:', METRICS.exists())
print('risk fact exist:', RISK.exists())

In [ ]:
if METRICS.exists():
    metrics = json.loads(METRICS.read_text())
    print('Model performance (test set):')
    for name, vals in metrics['models'].items():
        print(f"  {name:<22}  ROC-AUC={vals['roc_auc']:.3f}  PR-AUC={vals['pr_auc']:.3f}")
    print(f"\nF1-optimal threshold: {metrics['threshold']:.3f}")

In [ ]:
if RISK.exists():
    risk = pd.read_parquet(RISK)
    print('rows:', len(risk))
    print('\nrisk band distribution:')
    print(risk['RiskBand'].value_counts())
    print('\nten highest-risk employees:')
    cols = ['EmployeeID','RiskScore','RiskBand','TopDriver1','TopDriver2','TopDriver3']
    risk.sort_values('RiskScore', ascending=False)[cols].head(10)

## Limitations (read with the model card)

- Source is the small public IBM dataset; class balance is fixed.
- The time-series snapshots are synthetic — temporal validation is **not** performed.
- Attrition reason is unknown; the model only sees the flag.
- The model card lists out-of-scope uses; the fairness audit (Notebook 05) is the governance gate before any deployment talk.

## Interview talking points

- **Why two models** — the Logistic Regression baseline keeps stakeholders honest about the marginal value of the Random Forest, and gives an interpretable challenger model.
- **Why F1-optimal threshold** — the default 0.5 wastes signal on an imbalanced target. F1 keeps both precision and recall in the conversation.
- **Why SHAP** — global SHAP is the chart, but the per-employee top-3 drivers are what the Copilot actually narrates to a manager. That bridge from model to language is the product.